# Document와 Document Loaders

RAG(Retrieval-Augmented Generation) 시스템을 구축할 때 가장 먼저 해야 할 작업은 문서를 로드하는 것입니다.

LangChain은 다양한 형식의 문서를 일관된 방식으로 처리할 수 있도록 **Document 객체**와 **Document Loaders**를 제공합니다.

# 6-1. Document Loaders — RAG 파이프라인의 첫 관문

## 학습 목표
- RAG(검색 증강 생성) 파이프라인 전체 6단계 흐름을 이해해요
- `Document` 객체의 구조(page_content, metadata)를 파악해요
- 네 가지 로딩 방식(load / load_and_split / lazy_load / aload)의 차이를 설명할 수 있어요

## 사전 지식
- Python 기본 문법 (클래스, 딕셔너리)
- `langchain-core` 패키지 설치 완료

---

## RAG 파이프라인에서 Load 단계의 위치

아래 흐름도에서 지금 배우는 내용이 **어디에 해당하는지** 먼저 확인해 보세요.

```mermaid
flowchart LR
    A[📄 Load<br/>문서 로드]:::current --> B[✂️ Chunk<br/>청크 분할]:::process
    B --> C[🔢 Embed<br/>임베딩 변환]:::process
    C --> D[🗄️ Store<br/>벡터 저장]:::storage
    D --> E[🔍 Retrieve<br/>검색]:::process
    E --> F[💬 Generate<br/>답변 생성]:::output

    classDef current fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

> **지금 배우는 단계**: Load — 다양한 형식의 파일을 읽어 `Document` 객체로 변환해요.

## 학습 목표

- LangChain의 `Document` 객체 구조 이해
- Document Loaders의 역할과 주요 메서드 학습
- 다양한 로딩 방식(동기/비동기/지연 로딩) 비교

## Document 객체

`Document`는 LangChain에서 텍스트 데이터를 표현하는 기본 단위입니다.

### 주요 속성

- **`page_content`**: 문서의 실제 텍스트 내용 (문자열)
- **`metadata`**: 문서에 대한 추가 정보 (딕셔너리)
  - 예: 파일명, 페이지 번호, 작성자, 날짜 등

In [1]:
from langchain_core.documents import Document

# ---------------------------------------------------
# Document 객체 생성 및 기본 구조 확인
# ---------------------------------------------------

# 1단계: Document 객체 생성
# - page_content: 문서의 실제 텍스트 내용 (문자열)
# - metadata: 출처, 페이지 번호 등 부가 정보 (딕셔너리, 기본값: {})

# 아래에 코드를 작성하세요
document = Document(
    page_content="Langchain은 LLM을 활용한 앱 개발을 위한 프레임워크입니다."
)


In [2]:
# Document 객체의 속성 확인

# 아래에 코드를 작성하세요
print(document)
print(document.page_content)
print(document.metadata)


page_content='Langchain은 LLM을 활용한 앱 개발을 위한 프레임워크입니다.'
Langchain은 LLM을 활용한 앱 개발을 위한 프레임워크입니다.
{}


### 메타데이터 추가하기

Document 객체에 다양한 메타데이터를 추가할 수 있습니다. 메타데이터는 문서의 출처를 추적하거나 필터링할 때 유용합니다.

In [3]:
# ============================================================
# TODO: metadata 딕셔너리에 항목을 추가해 보세요
# 힌트: document.metadata["키"] = "값" 형식으로 추가합니다
# 예상 결과: metadata에 source, chapter, section, page 키가 생성됩니다
# ============================================================

# 1단계: 메타데이터 항목 추가
# - 메타데이터는 RAG 검색 시 필터링이나 소스 추적에 활용됨
document.metadata["source"] = "Langchain Tutorial"
document.metadata["chapter"] = "RAG"
document.metadata["section"] = "6-1. Document Loaders"
document.metadata["page"] = 1


In [4]:
# 업데이트된 메타데이터 확인

# 아래에 코드를 작성하세요
document.metadata

{'source': 'Langchain Tutorial',
 'chapter': 'RAG',
 'section': '6-1. Document Loaders',
 'page': 1}

---

> **섹션 전환** — Document 객체 구조를 익혔으니, 이제 실제로 파일을 읽어주는 **Document Loader**를 살펴볼게요.

## Document Loaders

Document Loaders는 다양한 형식의 파일(PDF, CSV, JSON, 웹 페이지 등)을 읽어서 `Document` 객체로 변환하는 역할을 합니다.

### 주요 Document Loaders

| Loader | 설명 | 용도 |
|--------|------|------|
| `PyPDFLoader` | PDF 파일 로딩 | 가장 많이 사용되는 로더 |
| `CSVLoader` | CSV 파일 로딩 | 테이블 데이터 처리 |
| `JSONLoader` | JSON 파일 로딩 | 구조화된 데이터 |
| `TextLoader` | 텍스트 파일 로딩 | 일반 텍스트 문서 |
| `WebBaseLoader` | 웹 페이지 로딩 | 온라인 콘텐츠 |
| `DirectoryLoader` | 디렉토리 일괄 로딩 | 여러 파일 동시 처리 |

## PyPDFLoader 사용 예제

가장 많이 사용되는 `PyPDFLoader`를 통해 Document Loader의 기본 사용법을 알아보겠습니다.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

# ============================================================
# TODO: PyPDFLoader로 PDF 파일을 로딩하는 코드를 완성하세요
# 힌트: PyPDFLoader(file_path)로 로더를 생성합니다
# 예상 결과: "✅ PDF Loader 생성 완료: ..." 메시지 출력
# ============================================================

# 1단계: PDF 파일 경로 지정
file_path = "./data/2026_gov.pdf"

# 2단계: PyPDFLoader 초기화
# - PyPDFLoader: 가장 기본적인 PDF 로더, 페이지별로 Document 생성
loader = PyPDFLoader(file_path)
loader


### 1. load() - 전체 문서 로딩

`load()` 메서드는 문서 전체를 한 번에 로드하여 `List[Document]` 형태로 반환합니다.

- **특징**: 모든 페이지를 메모리에 한 번에 로드
- **반환**: `List[Document]`
- **사용 시기**: 문서 크기가 작거나 전체 문서를 즉시 처리해야 할 때

In [7]:
# ============================================================
# TODO: loader.load()를 호출하여 PDF를 로딩하세요
# 힌트: docs = loader.load() 로 전체 문서를 한 번에 로드합니다
# 예상 결과: 총 페이지 수, 문서 타입, Document 타입이 출력됩니다
# ============================================================

# 1단계: PDF 전체 로딩
# - load(): 모든 페이지를 메모리에 한 번에 로드 → List[Document] 반환
docs = loader.load()
len(docs)

501

In [11]:
# 첫 번째 페이지 내용 확인

# 아래에 코드를 작성하세요
docs[0]
print(f' ==> [Line 4]: \033[38;2;17;169;224m[docs[0]]\033[0m({type(docs[0]).__name__}) = \033[38;2;152;63;43m{docs[0]}\033[0m')

docs[1]
print(f' ==> [Line 7]: \033[38;2;7;55;232m[docs[1]]\033[0m({type(docs[1]).__name__}) = \033[38;2;67;165;113m{docs[1]}\033[0m')

docs[2]
print(f' ==> [Line 10]: \033[38;2;84;45;1m[docs[2]]\033[0m({type(docs[2]).__name__}) = \033[38;2;80;229;79m{docs[2]}\033[0m')



 ==> [Line 4]: [docs[0]](Document) = page_content='발간등록번호
11-1053000-100001-09
이렇게 
달라집니다년부터2026' metadata={'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 0, 'page_label': '1'}
 ==> [Line 7]: [docs[1]](Document) = page_content='이렇게 
달라집니다년부터2026' metadata={'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 1, 'page_label': '2'}
 ==> [Line 10]: [docs[2]](Document) = page_content='본 책의 내용은 2025년 12월 22일  기준으로 작성되었습니다. 
수록된 내용 중 예정인 사안은 추후 변동될 수 있으니, 정확한 
확인을 위해서는 해당 페이지의 담당부서로 문의하시기 바랍니다.' metadata={'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDes

### 2. load_and_split() - 문서 로딩 + 분할

`load_and_split()` 메서드는 문서를 로드한 후 지정된 `TextSplitter`를 사용하여 자동으로 분할합니다.

- **특징**: 로딩과 분할을 한 번에 수행
- **반환**: `List[Document]` (분할된 청크들)
- **사용 시기**: 긴 문서를 작은 청크로 나눠야 할 때 (임베딩 생성 전처리)

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# TODO: load_and_split()을 사용하여 로딩과 분할을 한 번에 수행하세요
# 힌트: loader.load_and_split(text_splitter=splitter)
# 예상 결과: 원본 페이지 수보다 많은 청크가 생성됩니다
# ============================================================

# 1단계: 텍스트 분할기 설정
# - chunk_size=200: 한 청크의 최대 문자 수
# - chunk_overlap=0: 인접 청크 간 중복 없음

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

# 2단계: 로더 재초기화
loader = PyPDFLoader(file_path)

# 3단계: 문서 로딩 및 분할
# - load_and_split(): 로딩과 청크 분할을 한 번에 처리
split_docs = loader.load_and_split(text_splitter=text_splitter)

# 스플리터 사용 했을 때 안 했을 때 비교
print("원본:", len(docs))
print("청크:", len(split_docs))


원본: 501
청크: 2180


> **실무 팁**: 대용량 파일을 처리할 때는 `lazy_load()`를 사용하세요. 제너레이터 방식이라 수백 페이지 PDF도 메모리 부담 없이 스트리밍 처리할 수 있습니다.

In [21]:
# 분할된 청크 예시 확인

# 아래에 코드를 작성하세요
docs[4]
print(f' ==> [Line 4]: \033[38;2;191;46;244m[docs[4]]\033[0m({type(docs[4]).__name__}) = \033[38;2;40;237;128m{docs[4]}\033[0m')


split_docs[4]
split_docs[5]
print(f' ==> [Line 7]: \033[38;2;19;106;177m[split_docs[4]]\033[0m({type(split_docs[4]).__name__}) = \033[38;2;48;141;86m{split_docs[4]}\033[0m')
print(f' ==> [Line 9]: \033[38;2;18;219;186m[split_docs[5]]\033[0m({type(split_docs[5]).__name__}) = \033[38;2;97;51;137m{split_docs[5]}\033[0m')



 ==> [Line 4]: [docs[4]](Document) = page_content='분야별 달라지는 주요 제도
01 금융·재정·조세
통합고용세액공제 공제액 구조 개편 및 사후관리 합리화 재정경제부 008
웹툰콘텐츠 제작비용 세액공제 신설 재정경제부 009
자녀 수에 따라 보육수당 비과세 한도 확대 재정경제부 010
초등 저학년 예체능 학원비 세제지원 재정경제부 011
자녀 수에 따라 신용카드 등 소득공제 한도 확대 재정경제부 012
고배당기업에 대한 배당소득 분리과세 도입 재정경제부 013
증권거래세율 조정 재정경제부 014
상호금융 예탁금·출자금 비과세 적용기한 연장 및 적용범위 합리화 재정경제부 015
니코틴이 포함된 제품까지 법적 ‘담배’ 정의 확대 재정경제부 016
지정납부기한 이후 납부지연가산세 산출방법 개편 재정경제부 017
영세개인사업자 체납액 징수특례 확대 재정경제부 018
생계형 창업중소기업 세액감면 적용 기준금액 상향 재정경제부 019
영상콘텐츠 세제지원 확대 재정경제부 020
부분복귀하는 해외진출기업에 대한 소득세·법인세·관세 감면 확대 재정경제부 021
위기지역 창업기업 등에 대한 세액감면 합리화 재정경제부 022
지방이전 기업 세제지원 제도 개선 재정경제부 023
장애인 표준사업장에 대한 세액감면 확대 재정경제부 024
육아휴직급여·수당 비과세 대상 확대 재정경제부 025
대학생 교육비 특별세액공제 소득요건 폐지 재정경제부 026
상용근로자 간이지급명세서 월별 제출시기 유예 재정경제부 027
임목 벌채·양도소득 비과세 한도 확대 재정경제부 028
연금소득 원천징수세율 인하 재정경제부 029
AI 우수인력 등 국내 복귀 소득세 감면 적용기한 연장 재정경제부 030
고향사랑기부금 세액공제 확대 재정경제부 031
무주택 주말부부에 대해 각각 월세 세액공제 허용 재정경제부 032
상가임대료 인하 임대사업자에 대한 세액공제 적용기한 연장 재정경제부 033목차 2026년부터 이렇게 달라집니다 
https://whatsnew.moef.go.k

### 3. lazy_load() - 지연 로딩 (제너레이터)

`lazy_load()` 메서드는 제너레이터(generator) 방식으로 문서를 하나씩 로드합니다.

- **특징**: 메모리에 한 번에 하나의 Document만 로드
- **반환**: `Generator[Document]`
- **사용 시기**: 대용량 문서를 처리할 때, 메모리 절약이 중요할 때

In [42]:
# ============================================================
# TODO: lazy_load()로 제너레이터 방식 로딩을 해보세요
# 힌트: loader.lazy_load()는 Generator를 반환합니다
# 예상 결과: 반환 타입이 <class 'generator'>로 출력됩니다
# ============================================================

# 1단계: 로더 재초기화
loader = PyPDFLoader(file_path)

# 2단계: lazy_load() 호출 — 제너레이터 반환 (아직 메모리에 로드 안 됨)
# - lazy_load(): 한 번에 하나의 Document만 메모리에 로드 (메모리 절약)
lazy_docs = loader.lazy_load()
print(f' ==> [Line 12]: \033[38;2;204;119;228m[lazy_docs]\033[0m({type(lazy_docs).__name__}) = \033[38;2;70;76;44m{lazy_docs}\033[0m')



 ==> [Line 12]: [lazy_docs](generator) = <generator object PyPDFLoader.lazy_load at 0x12c823010>


In [43]:
# 제너레이터 순회 (한 번에 하나씩 로드)

# 아래에 코드를 작성하세요
for i, doc in enumerate(lazy_docs, start=1):
    if i == 10:
        break
    print(doc.metadata)
    print(doc.page_content)
    print("===============")


{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 0, 'page_label': '1'}
발간등록번호
11-1053000-100001-09
이렇게 
달라집니다년부터2026
{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 1, 'page_label': '2'}
이렇게 
달라집니다년부터2026
{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 2, 'page_label': '3'}
본 책의 내용은 2025년 12월 22일  기준으로 작성되었습니다. 
수록된 내용 중 예정인 사안은 추후 변동될 수 있으니,

**언제 lazy_load()를 사용하면 좋을까요?**

- ✅ **대용량 문서**: 수백 페이지 이상의 PDF 파일
- ✅ **메모리 제약**: 제한된 환경에서 실행할 때
- ✅ **스트리밍 처리**: 문서를 실시간으로 처리해야 할 때
- ❌ **전체 분석**: 모든 문서를 한 번에 분석해야 할 때는 `load()` 사용

### 4. aload() - 비동기 로딩

`aload()` 메서드는 비동기(async) 방식으로 문서를 로드합니다.

- **특징**: 비동기 프로그래밍 지원 (async/await)
- **반환**: `Awaitable[List[Document]]`
- **사용 시기**: 비동기 애플리케이션, 여러 문서를 동시에 로드할 때

## 핵심 정리 및 마무리

### Document 객체
- **`page_content`**: 문서의 실제 텍스트 내용
- **`metadata`**: 출처, 페이지 번호 등 메타정보

### Document Loader 메서드

| 메서드 | 반환 타입 | 사용 시기 |
|--------|-----------|----------|
| `load()` | `List[Document]` | 작은 문서, 전체 로딩 |
| `load_and_split()` | `List[Document]` | 청크 분할이 필요할 때 |
| `lazy_load()` | `Generator[Document]` | 대용량 문서, 메모리 절약 |
| `aload()` | `Awaitable[List[Document]]` | 비동기 처리, 병렬 로딩 |

### 실무 팁
> 대용량 PDF를 처리할 때는 `lazy_load()`를 사용해요. 메모리에 한 번에 올리지 않아서 수백 페이지 문서도 안정적으로 처리할 수 있어요.

---

## 다음 예고

`Document` 객체와 Loader의 기본을 익혔으니, 다음 노트북에서는 **파일 형식별 전용 Loader**를 다뤄볼게요.

- **01-PDF-Loader**: PyPDF, PDFPlumber, PyMuPDF 비교
- **02-HWP-Loader**: 한글 문서(.hwp) 로딩
- **03~13**: CSV, Excel, Word, PowerPoint, JSON, 웹 페이지 등